# 01 — Label Pre-processing

Thin notebook: it only **imports**, **calls** `src/label_export.py` + `src/label_analysis.py`, and **displays**.
It turns the labels into a table and audits them. It does not use embeddings.

**Input:** Patent-Labelling-Tools' 02a-postprocessed reviewed xlsxs (`taxonomy.wizard_excel_glob`) + the PatSeer metadata Excel (`taxonomy.metadata_excel`).
**Output** (under `taxonomy.output_dir`):
- `labels_v1.parquet` (+ `.csv` + column dictionary): one row per labelled architecture
- `label_analysis/`: `batch_summary.csv`, `duplicate_type_counts.csv`, `qc_issues.csv`, `coverage_table.csv`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.config_loader import load_config
from src import label_export as lex
from src import label_analysis as lan

cfg = load_config()
out_dir = Path(cfg['taxonomy']['output_dir']) / 'label_analysis'
out_dir.mkdir(parents=True, exist_ok=True)

## 1. Inspect the raw export schema
Lists the distinct (Section, Sub_Dimension, Field) triples in the export, before pivoting.

In [ ]:
schema = lex.inspect_exports(cfg)
display(schema)

## 2. Build and save the canonical wide table
One row per architecture. Columns: taxonomy fields, metadata and the main image path.

In [ ]:
canon, images = lex.build_canonical(cfg)
display(canon.head())

labels_dir = lex.save_canonical(canon, cfg)
print('wrote labels_v1.parquet to', labels_dir)

## 3. Per-batch reconciliation
Counts of reviewed, approved and disapproved patents, and duplicate-type counts, per batch file.

In [ ]:
long = lex.load_long(cfg)
batches = lan.batch_summary(long)
display(batches)
batches.to_csv(out_dir / 'batch_summary.csv', index=False)

dup_types = lan.duplicate_type_counts(long)
display(dup_types)
dup_types.to_csv(out_dir / 'duplicate_type_counts.csv', index=False)

## 4. Canonical-table QC and coverage

In [ ]:
qc_issues = lan.validate_canonical(canon)
display(qc_issues)
qc_issues.to_csv(out_dir / 'qc_issues.csv', index=False)

coverage = lan.coverage_table(canon, cfg)
display(coverage.head(40))
coverage.to_csv(out_dir / 'coverage_table.csv', index=False)

n_flag = int(coverage['below_min'].sum())
print(f"classes below min_class_count={cfg['taxonomy']['min_class_count']}: {n_flag} / {len(coverage)}")